# Ingest new histology

Takes one animal's whole-slide scans to reviewed GFP coverage per target:

1. **Export** each `.vsi` to per-section OME-TIFFs (skips sections already exported)
2. **Orient**: in napari, click the brain's **centre**, its **front**, and the **side the notch is on** for every section, then press **Done**
3. **Save** the clicks and check that none are missing or ambiguous
4. **Measure** GFP coverage in each target's ROI: the finetuned shape if saved, otherwise the template placed from your clicks. There is no automatic GFP fitting
5. **Review**: in napari, check the measured ROIs and coverage on each section, mark it ✓ or ✗, then press **Done**
6. **Result**: coverage per target from the sections that passed

Clicks and review verdicts are saved in `data/section_orientation.csv` (tracked in git). They are saved on every move between sections, so closing a window loses nothing.

Finetuning is not part of this notebook. To move or resize T1–T6 by hand, run `.venv/bin/python scripts/finetune_rois.py --mouse <N>` after step 3 and before Measure; it saves the shapes to `data/roi_locations.csv`, which Measure picks up.

Why the clicks are needed: `docs/histology-pipeline.md#mouse_02`. Kernel: **Python 3.12 (fus-research .venv)**.

In [1]:
%gui qt
from pathlib import Path

import pandas as pd

from fus import histology, orientation, rois

ROOT = histology.REPO_ROOT

# ---- parameters: set these for each new animal ------------------------------
MOUSE = 6
VSI = sorted((ROOT / f"data/histology/Mouse_{MOUSE:02d}").glob("*.vsi"))
OUT = ROOT / f"data/processed/histology/Mouse_{MOUSE:02d}/ds4"
# Template runs get their own files: mouse01_slots.csv is the GFP-fit benchmark.
SLOTS_CSV = ROOT / f"results/histology/mouse{MOUSE:02d}_template_slots.csv"
FIGURES = ROOT / f"results/histology/figures/mouse{MOUSE:02d}_template"

# Mouse 1's notch clicks tell the pipeline which side of the animal targets 1-3
# are on. They only need doing once; leave False once they're in the sheet.
INCLUDE_MOUSE01 = False
MOUSE01 = sorted((ROOT / "data/processed/histology/Mouse_01/ds4").glob("section_s*.ome.tif"))

print(f"Mouse {MOUSE}: {len(VSI)} scan(s)")
for v in VSI:
    print("  ", v.relative_to(ROOT))

Mouse 6: 4 scan(s)
   data/histology/Mouse_06/Image_Cntr 6.10.1_01.vsi
   data/histology/Mouse_06/Image_Cntr 6.10.2_01.vsi
   data/histology/Mouse_06/Image_Cntr 6.11.1_01.vsi
   data/histology/Mouse_06/Image_Cntr 6.11.2_01.vsi


## 1. Export

Olympus scans hold a label image, an overview, and one series per section. Each section is written at 4× downsampling (1.3 µm/px), about 20 s each. Mouse 1 was a single `Image.vsi`; Mouse 2 came as one `.vsi` per slide (`slideNN_` prefix); from Mouse 3 on each `.vsi` is one scan named `Image_Cntr_<mouse>.<well>.<slide>_<scan>` (prefix `w<well>sl<slide>_<scan>`), with three channels (no DAPI). `*_Overview.vsi` files are skipped.

In [2]:
# One prefix per .vsi: section (Mouse 1), slide01 (Mouse 2), w10sl1_02 (Mouse 3 on:
# Image_Cntr_<mouse>.<well>.<slide>_<scan>). Overview scans hold no sections.
for vsi in [v for v in VSI if not histology.is_overview(v)]:
    prefix = histology.slide_prefix(vsi)
    if list(OUT.glob(f"{prefix}_s*.ome.tif")):
        print(f"{vsi.name}: already exported, skipping")
        continue
    for path in histology.export_sections(vsi, OUT, prefix=prefix):
        print(f"{vsi.name}: wrote {path.name}")

SECTIONS = sorted(OUT.glob("*.ome.tif"))
print(f"\n{len(SECTIONS)} sections:", ", ".join(orientation.section_name(p) for p in SECTIONS))

Image_Cntr 6.10.1_01.vsi: already exported, skipping
Image_Cntr 6.10.2_01.vsi: already exported, skipping
Image_Cntr 6.11.1_01.vsi: already exported, skipping
Image_Cntr 6.11.2_01.vsi: already exported, skipping

12 sections: w10sl1_01_s2, w10sl1_01_s3, w10sl1_01_s4, w10sl1_01_s5, w10sl2_01_s2, w10sl2_01_s3, w11sl1_01_s2, w11sl1_01_s3, w11sl1_01_s4, w11sl1_01_s5, w11sl2_01_s2, w11sl2_01_s3


## 2. Orient — napari

A window opens with each section in turn: NeuN in grey, the GFP stain in green. For each section, click in order:

1. **centre** (cyan): the middle of the brain, on the midline
2. **front** (red): the front tip of the brain, opposite the cerebellum
3. **notch side** (yellow): out to the side the notch is on, roughly at a right angle to the front arrow

After each click the next dot is selected. Arrows run from the centre, and the **yellow circles are the target template**. It's a rigid copy of the six-target plan at a fixed size, sitting ~1.6 mm in front of your centre and turned to your front arrow. Move the centre or front dot until it sits on the targets. Labels become T1–T6 once the notch side is set (Mouse 1's notch clicks are needed for that). **Space** saves and moves on. Tick **Exclude** for sections with no usable GFP, or that are torn or partial. When you've finished, press **Done** to save and close the window, then run the next cell.

Sections already clicked are loaded, so you can check or adjust them. Clicked sections are shown **straightened**: front up, notch side left, mirror-flipped if the section lies face down. This is a display transform only; the pixels are not resampled. New sections stay as scanned so the view doesn't turn while you click. Press **R** to straighten after clicking. The review window (and `scripts/finetune_rois.py`) are always straightened.

In [3]:
to_orient = SECTIONS + (MOUSE01 if INCLUDE_MOUSE01 else [])
annotator = orientation.annotate(to_orient)

## 3. Save the selections

Reads back what was clicked, flags anything missing or ambiguous, and records the table in this notebook.

In [4]:
sheet = orientation.read_sheet()
names = [orientation.section_name(p) for p in SECTIONS]

rows = []
for name in names:
    a = sheet.get((MOUSE, name), orientation.Annotation(MOUSE, name))
    rows.append({
        "section": name,
        "front points": orientation._compass(a.angle_deg) if a.angle_deg is not None else "",
        "front angle": None if a.angle_deg is None else round(a.angle_deg),
        "notch arrow vs front": None if a.notch_offset_deg is None else f"{a.notch_offset_deg:.0f}°",
        "exclude": a.exclude,
        "complete": a.complete,
        "notes": a.notes,
    })
selections = pd.DataFrame(rows).set_index("section")
display(selections)

missing = selections.index[~selections.complete].tolist()
ambiguous = [n for n in names if (o := sheet.get((MOUSE, n))) and o.notch_offset_deg is not None
             and abs(o.notch_offset_deg - 90) > 60]
plus_x = orientation.plus_x_side(sheet)
print("missing clicks:", ", ".join(missing) or "none")
print("notch-side arrow too close to the midline:", ", ".join(ambiguous) or "none")
print("targets 1-3 are on the animal's", plus_x or "?? (click the Mouse 1 sections: INCLUDE_MOUSE01 = True)")
assert not missing and not ambiguous, "fix these in the annotator (re-run the cell above) before measuring"

,front points,front angle,notch arrow vs front,exclude,complete,notes
section,,,,,,
w10sl1_01_s2,,NaN,NaN,True,True,
w10sl1_01_s3,down,14.0,89°,False,True,
w10sl1_01_s4,up-left,114.0,93°,False,True,
w10sl1_01_s5,,NaN,NaN,True,True,
w10sl2_01_s2,,NaN,NaN,True,True,
w10sl2_01_s3,,NaN,NaN,True,True,
w11sl1_01_s2,right,263.0,92°,False,True,
w11sl1_01_s3,,NaN,NaN,True,True,
w11sl1_01_s4,up-left,153.0,89°,False,True,


missing clicks: none
notch-side arrow too close to the midline: none
targets 1-3 are on the animal's right


## 4. Measure

GFP coverage in each target's ROI. That's the finetuned shape where one was saved by `scripts/finetune_rois.py`, otherwise the 1.0 mm template circle exactly where the orientation window drew it. Nothing is fitted to the GFP automatically. It takes about 12 s per section.

In [7]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

FIGURES.mkdir(parents=True, exist_ok=True)
locations = rois.read_locations()
tables = []
for path in SECTIONS:
    a = sheet[(MOUSE, orientation.section_name(path))]
    if a.exclude:
        print(f"{a.section}: excluded")
        continue
    sec = histology.load_section(path)
    template = histology.template_fit(a.centre, a.angle_deg, sec.pixel_um)
    finetuned = locations.get((MOUSE, sec.name))
    res = histology.analyse_section(sec, template=template, rois=finetuned)
    tables.append(res.table())
    fig = histology.plot_section(sec, res)
    fig.savefig(FIGURES / f"{sec.name}_slots.png", dpi=110)
    plt.close(fig)
    cov = {r["slot"]: r["coverage"] for r in res.rows if r["channel"] == "TRITC"}
    how = "finetuned" if finetuned else "template "
    print(f"{sec.name}: {how}  TRITC by slot " + " ".join(f"{cov[s]:.2f}" for s in sorted(cov)))
    del sec, res

slots = pd.concat(tables, ignore_index=True)
slots.to_csv(SLOTS_CSV, index=False)
print(f"\nwrote {len(slots)} rows to {SLOTS_CSV.relative_to(ROOT)}")

w10sl1_01_s2: excluded
w10sl1_01_s3: finetuned  TRITC by slot 0.05 0.01 0.11 0.24 0.17 0.35
w10sl1_01_s4: finetuned  TRITC by slot 0.16 0.07 0.55 0.56 0.09 0.75
w10sl1_01_s5: excluded
w10sl2_01_s2: excluded
w10sl2_01_s3: excluded
w11sl1_01_s2: finetuned  TRITC by slot 0.07 0.31 0.01 0.03 0.03 0.00
w11sl1_01_s3: excluded
w11sl1_01_s4: finetuned  TRITC by slot 0.16 0.01 0.11 0.39 0.24 0.56
w11sl1_01_s5: finetuned  TRITC by slot 0.50 0.01 0.51 0.37 0.11 0.62
w11sl2_01_s2: finetuned  TRITC by slot 0.10 0.22 0.05 0.02 0.02 0.00
w11sl2_01_s3: finetuned  TRITC by slot 0.00 0.02 0.09 0.03 0.12 0.30

wrote 84 rows to results/histology/mouse06_template_slots.csv


## 5. Review — napari

A window shows each measured section with its ROIs exactly as measured (finetuned shapes or template circles), labelled **T1–T6** with TRITC coverage, plus your arrows. For each section, check that each ROI sits on its target and the labels are on the right side, then press:

- **Looks right ✓** (Y), or
- **Wrong ✗** (X). Wrong sections are left out of the result.

Press **Done** when finished, then run the last cell.

In [8]:
reviewer = orientation.review([p for p in SECTIONS
                               if not sheet[(MOUSE, orientation.section_name(p))].exclude],
                              pd.read_csv(SLOTS_CSV))

## 6. Coverage per target

This uses only sections that are clicked, not excluded, and marked ✓ in the review. Values are TRITC (anti-GFP stain) coverage in the 1.0 mm ROI. Joining these to the acoustic dose is `scripts/mouse02_dose_vs_coverage.py` (for Mouse 2).

In [9]:
sheet = orientation.read_sheet()
plus_x = orientation.plus_x_side(sheet)
slots = pd.read_csv(SLOTS_CSV)

verdicts = pd.Series({n: sheet[(MOUSE, n)].review or "not reviewed" for n in names}, name="review")
display(verdicts.to_frame())

good = [n for n in names
        if sheet[(MOUSE, n)].review == "ok"
        and not slots.loc[slots.section == n, "angle_flag"].any()]
d = slots[slots.section.isin(good) & (slots.channel == "TRITC")].copy()
d["target"] = [histology.slot_to_target(s, orientation.is_mirrored(sheet[(MOUSE, n)], plus_x))
               for s, n in zip(d.slot, d.section)]

per_section = d.pivot(index="section", columns="target", values="coverage").round(2)
print(f"{len(good)} of {len(names)} sections used")
display(per_section)
summary = d.groupby("target").coverage.agg(["mean", "min", "max", "count"]).round(2)
display(summary)

,review
w10sl1_01_s2,not reviewed
w10sl1_01_s3,not reviewed
w10sl1_01_s4,not reviewed
w10sl1_01_s5,not reviewed
w10sl2_01_s2,not reviewed
w10sl2_01_s3,not reviewed
w11sl1_01_s2,not reviewed
w11sl1_01_s3,not reviewed
w11sl1_01_s4,not reviewed
w11sl1_01_s5,not reviewed


0 of 12 sections used


target
section


,mean,min,max,count
target,,,,
